In [70]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
# Настраиваем визуальное оформление графиков
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Папка для сохранения графиков
from pathlib import Path; Path('eda_plots').mkdir(parents=True, exist_ok=True)

In [71]:
# Загружаем файл с данными
df = pd.read_excel('data.xlsx')

# Удаляем служебный столбец-индекс
df = df.drop(columns=['Unnamed: 0'])

print(f"\n Размер датасета: {df.shape}")
print(f" Количество объектов : {df.shape[0]}")
print(f" Количество признаков (включая целевые): {df.shape[1]}")

# Список целевых переменных
target_cols = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in df.columns if col not in target_cols]
print(f" Целевые переменные: {target_cols}")
print(f" Количество признаков (без целевых): {len(feature_cols)}")


 Размер датасета: (1001, 213)
 Количество объектов : 1001
 Количество признаков (включая целевые): 213
 Целевые переменные: ['IC50, mM', 'CC50, mM', 'SI']
 Количество признаков (без целевых): 210


In [72]:
# Анализ пропусков и типов данных

# Проверяем типы данных
print("\nТипы данных:")
print(df.dtypes.value_counts())
# Ищем пропуски
missing = df.isnull().sum()
missing = missing[missing > 0]

print(f"\nКоличество столбцов с пропусками: {len(missing)}")
if len(missing) > 0:
    print("\nСписок столбцов с пропусками:")
    for col, cnt in missing.items():
        # Считаем процент пропусков
        pct = cnt / len(df) * 100
        print(f"  - {col}: {cnt} пропусков ({pct:.2f}%)")
else:
    print("Пропусков в данных нет!")


Типы данных:
float64    107
int64      106
Name: count, dtype: int64

Количество столбцов с пропусками: 12

Список столбцов с пропусками:
  - MaxPartialCharge: 3 пропусков (0.30%)
  - MinPartialCharge: 3 пропусков (0.30%)
  - MaxAbsPartialCharge: 3 пропусков (0.30%)
  - MinAbsPartialCharge: 3 пропусков (0.30%)
  - BCUT2D_MWHI: 3 пропусков (0.30%)
  - BCUT2D_MWLOW: 3 пропусков (0.30%)
  - BCUT2D_CHGHI: 3 пропусков (0.30%)
  - BCUT2D_CHGLO: 3 пропусков (0.30%)
  - BCUT2D_LOGPHI: 3 пропусков (0.30%)
  - BCUT2D_LOGPLOW: 3 пропусков (0.30%)
  - BCUT2D_MRHI: 3 пропусков (0.30%)
  - BCUT2D_MRLOW: 3 пропусков (0.30%)


In [73]:
# Статистический анализ целевых переменных

# Базовая статистика
print("\nОписательная статистика для целевых переменных:")
print(df[target_cols].describe())

# Проверяем, что SI = CC50 / IC50 (как указано в задании)
calculated_si = df['CC50, mM'] / df['IC50, mM']
diff = (df['SI'] - calculated_si).abs()
print(f"Максимальное расхождение: {diff.max():.6f}")
print(f"Среднее расхождение: {diff.mean():.6f}")
# Если расхождение мало,то формула подтверждена
if (diff < 0.01).all():
    print("Вывод: формула SI = CC50 / IC50 полностью подтверждается")
else:
    print("Вывод: формула SI = CC50 / IC50 неправильная")


Описательная статистика для целевых переменных:
          IC50, mM     CC50, mM            SI
count  1001.000000  1001.000000   1001.000000
mean    222.805156   589.110728     72.508823
std     402.169734   642.867508    684.482739
min       0.003517     0.700808      0.011489
25%      12.515396    99.999036      1.433333
50%      46.585183   411.039342      3.846154
75%     224.975928   894.089176     16.566667
max    4128.529377  4538.976189  15620.600000
Максимальное расхождение: 0.000000
Среднее расхождение: 0.000000
Вывод: формула SI = CC50 / IC50 полностью подтверждается


In [74]:
# Визуализация распределений целевых переменных

#Строим гистограммы: исходные значения и логарифмированные
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, col in enumerate(target_cols):
    # Исходное распределение
    axes[0, i].hist(df[col], bins=50, color='steelblue', edgecolor='black')
    axes[0, i].set_title(f'Распределение {col} (исходное)')
    axes[0, i].set_xlabel(col)
    axes[0, i].set_ylabel('Частота')
    #Добавляем вертикальную линию для медианы
    axes[0, i].axvline(df[col].median(), color='red', linestyle='--',
                       label=f'медиана = {df[col].median():.2f}')
    axes[0, i].legend()

    # Логарифмированное распределение
    axes[1, i].hist(np.log1p(df[col]), bins=50, color='coral', edgecolor='black')
    axes[1, i].set_title(f'Распределение log(1 + {col})')
    axes[1, i].set_xlabel(f'log(1 + {col})')
    axes[1, i].set_ylabel('Частота')

plt.tight_layout()
plt.savefig('eda_plots/01_target_distributions.png', bbox_inches='tight')
plt.close()

# Строим boxplot для визуализации выбросов
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(target_cols):
    axes[i].boxplot(df[col], vert=True, patch_artist=True,
                     boxprops=dict(facecolor='lightblue'))
    axes[i].set_title(f'Boxplot для {col}')
    axes[i].set_ylabel(col)
plt.tight_layout()
plt.savefig('eda_plots/02_target_boxplots.png', bbox_inches='tight')
plt.close()

In [75]:
# Анализ выбросов (IQR)

# Метод межквартильного размаха для определения выбросов

for col in target_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    print(f"{col}:")
    print(f"  Q1 = {Q1:.3f}, Q3 = {Q3:.3f}, IQR = {IQR:.3f}")
    print(f"Границы: [{lower_bound:.3f}, {upper_bound:.3f}]")
    print(f"Количество выбросов: {outliers_count} ({outliers_count/len(df)*100:.2f}%)")
    print()

IC50, mM:
  Q1 = 12.515, Q3 = 224.976, IQR = 212.461
Границы: [-306.175, 543.667]
Количество выбросов: 147 (14.69%)

CC50, mM:
  Q1 = 99.999, Q3 = 894.089, IQR = 794.090
Границы: [-1091.136, 2085.224]
Количество выбросов: 39 (3.90%)

SI:
  Q1 = 1.433, Q3 = 16.567, IQR = 15.133
Границы: [-21.267, 39.267]
Количество выбросов: 125 (12.49%)



In [76]:
# Анализ для задач классификации (баланс классов)

for col in target_cols:
    median_val = df[col].median()
    # Создаем бинарный признак: 1 - выше медианы 0 - ниже
    above = (df[col] > median_val).sum()
    below = (df[col] <= median_val).sum()
    print(f"{col} > медианы ({median_val:.3f}):")
    print(f"  Класс 1 (выше): {above} ({above/len(df)*100:.1f}%)")
    print(f"  Класс 0 (ниже): {below} ({below/len(df)*100:.1f}%)")
    print()

# Отдельная задача: SI > 8
above_8 = (df['SI'] > 8).sum()
below_8 = (df['SI'] <= 8).sum()
print(f"SI > 8:")
print(f"  Класс 1 (SI > 8): {above_8} ({above_8/len(df)*100:.1f}%)")
print(f"  Класс 0 (SI <= 8): {below_8} ({below_8/len(df)*100:.1f}%)")

IC50, mM > медианы (46.585):
  Класс 1 (выше): 500 (50.0%)
  Класс 0 (ниже): 501 (50.0%)

CC50, mM > медианы (411.039):
  Класс 1 (выше): 499 (49.9%)
  Класс 0 (ниже): 502 (50.1%)

SI > медианы (3.846):
  Класс 1 (выше): 500 (50.0%)
  Класс 0 (ниже): 501 (50.0%)

SI > 8:
  Класс 1 (SI > 8): 357 (35.7%)
  Класс 0 (SI <= 8): 644 (64.3%)


In [77]:
#Корреляционный анализ

# заполняем пропуски
df_filled = df.copy()
for col in df_filled.columns:
    if df_filled[col].isnull().sum() > 0:
        # Заполняем пропуски медианой
        df_filled[col] = df_filled[col].fillna(df_filled[col].median())

# Считаем корреляции признаков с целевыми переменными
correlations = df_filled[feature_cols + target_cols].corr()[target_cols]
correlations = correlations.drop(target_cols)

# 15 признаков, наиболее коррелированных с каждой целевой переменной
print("\n15 признаков по корреляции с целевыми переменными:")
for col in target_cols:
    print(f"\n--- Корреляция с {col} ---")
    top_corr = correlations[col].abs().sort_values(ascending=False).head(15)
    for feat, corr_val in top_corr.items():
        # Получаем знак исходной корреляции
        original = correlations.loc[feat, col]
        print(f"  {feat:30s}: {original:+.4f}")

# Тепловая карта для топ-30 признаков 
all_top_features = set()
for col in target_cols:
    top_15 = correlations[col].abs().sort_values(ascending=False).head(15).index.tolist()
    all_top_features.update(top_15)

# Строим тепловую карту корреляций
top_features = list(all_top_features)
plt.figure(figsize=(8, 12))
heatmap_data = correlations.loc[top_features].sort_values(by='IC50, mM', ascending=False)
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            cbar_kws={'label': 'Коэффициент корреляции Пирсона'})
plt.title('Корреляция топ-признаков с целевыми переменными')
plt.tight_layout()
plt.savefig('eda_plots/03_correlation_heatmap.png', bbox_inches='tight')
plt.close()



15 признаков по корреляции с целевыми переменными:

--- Корреляция с IC50, mM ---
  VSA_EState4                   : -0.2742
  Chi2n                         : -0.2571
  PEOE_VSA7                     : -0.2560
  Chi2v                         : -0.2492
  fr_Ar_NH                      : +0.2455
  fr_Nhpyrrole                  : +0.2455
  Chi4v                         : -0.2436
  Chi4n                         : -0.2435
  Chi3n                         : -0.2397
  Chi3v                         : -0.2378
  SlogP_VSA5                    : -0.2364
  Chi1n                         : -0.2298
  Chi1v                         : -0.2198
  MolLogP                       : -0.2184
  fr_nitro                      : +0.2159

--- Корреляция с CC50, mM ---
  MolMR                         : -0.3101
  LabuteASA                     : -0.3092
  MolWt                         : -0.3064
  ExactMolWt                    : -0.3064
  HeavyAtomCount                : -0.3052
  Chi0                          : -0.3048
  Ch

In [78]:
# Анализ взаимной корреляции признаков

# Считаем матрицу корреляций между признаками
feature_corr = df_filled[feature_cols].corr().abs()

# Берем только верхний треугольник матрицы (без диагонали)
upper_tri = feature_corr.where(
    np.triu(np.ones(feature_corr.shape), k=1).astype(bool)
)

# Ищем пары с очень высокой корреляцией >0.95
high_corr_pairs = []
for col in upper_tri.columns:
    high = upper_tri[col][upper_tri[col] > 0.95]
    for idx, val in high.items():
        high_corr_pairs.append((col, idx, val))

print(f"\nПар признаков с корреляцией > 0.95: {len(high_corr_pairs)}")

if len(high_corr_pairs) > 0:
    print("\n10 пар с наибольшей корреляцией:")
    high_corr_pairs.sort(key=lambda x: x[2], reverse=True)
    for f1, f2, val in high_corr_pairs[:10]:
        print(f"  {f1:30s} <-> {f2:30s}: {val:.4f}")


Пар признаков с корреляцией > 0.95: 91

10 пар с наибольшей корреляцией:
  MaxEStateIndex                 <-> MaxAbsEStateIndex             : 1.0000
  fr_COO2                        <-> fr_COO                        : 1.0000
  fr_Nhpyrrole                   <-> fr_Ar_NH                      : 1.0000
  fr_benzene                     <-> NumAromaticCarbocycles        : 1.0000
  fr_phenol_noOrthoHbond         <-> fr_phenol                     : 1.0000
  ExactMolWt                     <-> MolWt                         : 1.0000
  HeavyAtomCount                 <-> Chi1                          : 0.9987
  HeavyAtomMolWt                 <-> MolWt                         : 0.9969
  ExactMolWt                     <-> HeavyAtomMolWt                : 0.9968
  HeavyAtomCount                 <-> Chi0                          : 0.9960


In [79]:
# Проверка константных признаков
constant_features = []
near_constant_features = []

for col in feature_cols:
    n_unique = df_filled[col].nunique()
    if n_unique == 1:
        constant_features.append(col)
    elif n_unique > 1:
        # Проверяем, не доминирует ли одно значение
        top_freq = df_filled[col].value_counts(normalize=True).iloc[0]
        if top_freq > 0.99:
            near_constant_features.append((col, top_freq))

print(f"\nКонстантных признаков: {len(constant_features)}")
if constant_features:
    print(f"Список: {constant_features}")

print(f"\nПочти константных признаков: {len(near_constant_features)}")
if near_constant_features:
    print("Список:")
    for col, freq in near_constant_features[:20]:
        print(f"  {col:30s}: доминирующее значение в {freq*100:.2f}%")


Константных признаков: 18
Список: ['NumRadicalElectrons', 'SMR_VSA8', 'SlogP_VSA9', 'fr_N_O', 'fr_SH', 'fr_azide', 'fr_barbitur', 'fr_benzodiazepine', 'fr_diazo', 'fr_dihydropyridine', 'fr_isocyan', 'fr_isothiocyan', 'fr_lactam', 'fr_nitroso', 'fr_phos_acid', 'fr_phos_ester', 'fr_prisulfonamd', 'fr_thiocyan']

Почти константных признаков: 15
Список:
  fr_Ar_COO                     : доминирующее значение в 99.90%
  fr_HOCCN                      : доминирующее значение в 99.90%
  fr_aldehyde                   : доминирующее значение в 99.70%
  fr_amidine                    : доминирующее значение в 99.20%
  fr_azo                        : доминирующее значение в 99.30%
  fr_epoxide                    : доминирующее значение в 99.60%
  fr_guanido                    : доминирующее значение в 99.60%
  fr_hdrzine                    : доминирующее значение в 99.70%
  fr_nitrile                    : доминирующее значение в 99.40%
  fr_oxazole                    : доминирующее значение в 99.6

In [80]:
# Визуализация связи между IC50, CC50 И SI

# Используем логарифмированные значения для лучшей визуализации
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# IC50 vs CC50
axes[0].scatter(np.log1p(df['IC50, mM']), np.log1p(df['CC50, mM']),
                alpha=0.5, color='steelblue')
axes[0].set_xlabel('log(1 + IC50)')
axes[0].set_ylabel('log(1 + CC50)')
axes[0].set_title('Связь между IC50 и CC50')

# IC50 vs SI
axes[1].scatter(np.log1p(df['IC50, mM']), np.log1p(df['SI']),
                alpha=0.5, color='coral')
axes[1].set_xlabel('log(1 + IC50)')
axes[1].set_ylabel('log(1 + SI)')
axes[1].set_title('Связь между IC50 и SI')

# CC50 vs SI
axes[2].scatter(np.log1p(df['CC50, mM']), np.log1p(df['SI']),
                alpha=0.5, color='green')
axes[2].set_xlabel('log(1 + CC50)')
axes[2].set_ylabel('log(1 + SI)')
axes[2].set_title('Связь между CC50 и SI')

plt.tight_layout()
plt.savefig('eda_plots/04_target_relations.png', bbox_inches='tight')
plt.close()